# Итоговое решение AMML ranking competition 2026
# Обучение и инференс выполнены в Google Colab (Linux, NVIDIA Tesla T4).

## Установка и импорты

In [2]:
import gc
import html
import os
import re
import unicodedata
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from google.colab import files
from lightgbm import LGBMRanker
from threadpoolctl import threadpool_limits
from tqdm.auto import tqdm

os.environ.setdefault("HF_HUB_DOWNLOAD_TIMEOUT", "120")

from transformers import AutoModel, AutoTokenizer

## Исходные данные

In [3]:
OUTPUT_DIR = Path("/content")
missing_files = [
    name for name in ("train.csv", "test.csv") if not (OUTPUT_DIR / name).is_file()
]
if missing_files:
    raise FileNotFoundError(f"Нет файлов в {OUTPUT_DIR}: {', '.join(missing_files)}")

pair_types = {"id": str, "query_id": str}
train = pd.read_csv(OUTPUT_DIR / "train.csv", dtype=pair_types)
test = pd.read_csv(OUTPUT_DIR / "test.csv", dtype=pair_types)

pair_columns = {
    "id", "query_id", "article_id", "query", "title", "text",
    "passage_index", "article_passage_count", "article_char_count", "ingest_source",
}
for frame in (train, test):
    missing = pair_columns - set(frame.columns)
    if missing:
        raise ValueError(f"Отсутствуют столбцы {sorted(missing)}")
    if not frame.id.is_unique or frame[["id", "query_id"]].isna().any().any():
        raise ValueError("Проверьте уникальность id и пропуски в id и query_id.")
    if not frame.groupby("query_id")["query"].nunique(dropna=False).eq(1).all():
        raise ValueError("Каждому query_id должен соответствовать один текст запроса.")

if "relevance" not in train or not train.relevance.isin([0, 1]).all():
    raise ValueError("В Train нужны метки relevance со значениями 0 и 1.")
if set(train.query_id) & set(test.query_id):
    raise ValueError("В Train и Test найдены общие query_id.")

print(f"Train: {len(train):,} строк. Test: {len(test):,} строк.")


Train: 31,530 строк. Test: 13,507 строк.


## Скоры Jina на Train

In [4]:
MODEL_NAME = "jinaai/jina-reranker-v3.5"
MODEL_REVISION = "d666730b5ddaae829437e38f5f117b456e81d1d2"
DOC_CHAR_LIMIT = 1800


def clean_string(value):
    if pd.isna(value):
        return ""
    text = str(value)
    if text.strip().lower() in {"nan", "none", "null", "n/a", "na", "<na>"}:
        return ""
    text = unicodedata.normalize("NFC", html.unescape(text))
    text = re.sub(r"<[^>]+>", " ", text)
    return re.sub(r"\s+", " ", text).strip()


def jina_doc(row):
    title = clean_string(row.title) or "Без заголовка"
    body = clean_string(row.text)[:DOC_CHAR_LIMIT]
    return f"Заголовок: {title}\nФрагмент: {body}"


def load_jina():
    if not torch.cuda.is_available():
        raise RuntimeError("Требуется GPU с CUDA.")

    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print("Загрузка весов Jina…", flush=True)
    model = AutoModel.from_pretrained(
        MODEL_NAME,
        revision=MODEL_REVISION,
        trust_remote_code=True,
        attn_implementation="sdpa",
        dtype=torch.float16,
    ).to("cuda").eval()
    print("Загрузка токенизатора Jina…", flush=True)
    tokenizer = AutoTokenizer.from_pretrained(
        MODEL_NAME, revision=MODEL_REVISION, trust_remote_code=True
    )
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.unk_token
        tokenizer.pad_token_id = tokenizer.convert_tokens_to_ids(tokenizer.pad_token)
    tokenizer.padding_side = "left"
    model._tokenizer = tokenizer
    return model


def jina_scores(frame, model, split_name=None):
    if split_name is None:
        split_name = "Train" if "relevance" in frame.columns else "Test"

    groups = frame.sort_values(["query_id", "id"], kind="stable").groupby(
        "query_id", sort=False
    )
    scores = []

    with torch.inference_mode():
        for reverse in (False, True):
            by_id = {}
            order = "обратный порядок" if reverse else "прямой порядок"

            progress = tqdm(
                groups,
                total=groups.ngroups,
                desc=f"Jina {split_name} ({order})",
                unit="запрос",
                mininterval=2,
            )

            for _, group in progress:
                ids = group.id.tolist()
                docs = [jina_doc(row) for row in group.itertuples(index=False)]

                if reverse:
                    ids.reverse()
                    docs.reverse()

                result = model.rerank(
                    clean_string(group["query"].iat[0]), docs, top_n=len(docs)
                )
                by_position = {
                    int(row["index"]): float(row["relevance_score"])
                    for row in result
                }

                if len(result) != len(ids) or set(by_position) != set(range(len(ids))):
                    raise ValueError("Jina вернула неполный список кандидатов.")

                by_id.update(
                    {key: by_position[position] for position, key in enumerate(ids)}
                )

            values = frame.id.map(by_id).to_numpy(dtype=float)
            if not np.isfinite(values).all():
                raise ValueError("Найдены пропуски или некорректные скоры Jina.")

            scores.append(values)

    return tuple(scores)

In [5]:
jina = load_jina()
train_forward, train_reverse = jina_scores(train, jina, "Train")

GPU: Tesla T4
Загрузка весов Jina…


config.json:   0%|          | 0.00/1.52k [00:00<?, ?B/s]

modeling.py:   0%|          | 0.00/10.4k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.19GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/312 [00:00<?, ?it/s]

... (7 outputs left out)

added_tokens.json:   0%|          | 0.00/795 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/777 [00:00<?, ?B/s]

Jina Train (прямой порядок):   0%|          | 0/3042 [00:00<?, ?запрос/s]

Jina Train (обратный порядок):   0%|          | 0/3042 [00:00<?, ?запрос/s]

## Признаки ранкеров

In [6]:
TOKEN_RE = re.compile(r"(?u)[a-zа-яё0-9]+")
DIGIT_RE = re.compile(r"(?u)\b\d+\b")


def build_base_features(frame, forward):
    rows = frame.reset_index(drop=True)
    normalized = {
        name: rows[name].map(clean_string).str.lower().str.replace("ё", "е", regex=False)
        for name in ("query", "title", "text")
    }
    tokens = {
        name: text.map(lambda value: set(TOKEN_RE.findall(value)))
        for name, text in normalized.items()
    }

    f = pd.DataFrame({"jina_score": np.asarray(forward, dtype=float)})
    group = f.jina_score.groupby(rows.query_id, sort=False)
    f["jina_group_n"] = group.transform("size")
    f["jina_group_mean"] = group.transform("mean")
    f["jina_group_std"] = group.transform("std").fillna(0)
    f["jina_group_max"] = group.transform("max")
    f["jina_rank"] = group.rank(method="min", ascending=False)
    f["jina_rank_pct"] = f.jina_rank / f.jina_group_n
    f["jina_gap_to_top"] = f.jina_score - f.jina_group_max
    f["jina_centered"] = f.jina_score - f.jina_group_mean
    f["jina_z"] = f.jina_centered / (f.jina_group_std + 1e-3)

    for name in ("query", "title", "text"):
        f[f"{name}_chars"] = normalized[name].str.len()
        f[f"{name}_words"] = tokens[name].map(len)

    title_overlap = [len(q & t) for q, t in zip(tokens["query"], tokens["title"])]
    text_overlap = [len(q & t) for q, t in zip(tokens["query"], tokens["text"])]
    f["title_overlap"] = title_overlap
    f["text_overlap"] = text_overlap
    f["title_coverage"] = f.title_overlap / np.maximum(f.query_words, 1)
    f["text_coverage"] = f.text_overlap / np.maximum(f.query_words, 1)

    query_numbers = normalized["query"].map(lambda value: set(DIGIT_RE.findall(value)))
    doc_numbers = (normalized["title"] + " " + normalized["text"]).map(
        lambda value: set(DIGIT_RE.findall(value))
    )
    number_overlap = np.array([len(q & d) for q, d in zip(query_numbers, doc_numbers)])
    f["number_coverage"] = number_overlap / np.maximum(query_numbers.map(len), 1)
    f["title_missing"] = f.title_chars.eq(0).astype(np.int8)
    f["text_missing"] = f.text_chars.eq(0).astype(np.int8)

    for name in ("passage_index", "article_passage_count", "article_char_count"):
        raw = pd.to_numeric(rows[name], errors="coerce").mask(lambda values: values < 0)
        f[f"{name}_missing"] = raw.isna().astype(np.int8)
        f[name] = raw.fillna(-1)

    f["passage_pos_ratio"] = np.where(
        (f.article_passage_count > 0) & (f.passage_index >= 0),
        f.passage_index / np.maximum(f.article_passage_count, 1),
        -1,
    )
    f["same_article_in_query"] = rows.id.groupby(
        [rows.query_id, rows.article_id.fillna("__missing__")], sort=False
    ).transform("size")
    f["source"] = rows.ingest_source.fillna("MISSING").astype(str)
    f = pd.get_dummies(f, columns=["source"], dtype=np.int8).astype(np.float32)
    assert np.isfinite(f.to_numpy()).all()
    return f


def build_article_features(frame, forward, reverse):
    keys = [frame.query_id, frame.article_id.fillna("")]
    fwd = pd.Series(forward).groupby(keys, sort=False)
    rev = pd.Series(reverse).groupby(keys, sort=False)
    count = fwd.transform("size")
    mean = fwd.transform("mean")
    maximum = fwd.transform("max")
    reverse_max = rev.transform("max")

    f = pd.DataFrame({
        "article_mean": mean,
        "article_top": maximum,
        "article_rank": fwd.rank(ascending=False, method="min"),
        "article_gap": forward - maximum,
        "article_second": count.ge(2).astype(int) * (mean * count - forward)
                          / np.maximum(count - 1, 1),
        "article_reverse_top": reverse_max,
        "article_reverse_gap": reverse - reverse_max,
        "article_reverse_rank": rev.rank(ascending=False, method="min"),
    }).astype(np.float32)
    assert np.isfinite(f.to_numpy()).all()
    return f



def build_branch(frame, primary, complementary, base_columns=None):
    base = build_base_features(frame, primary)
    if base_columns is not None:
        base = base.reindex(columns=base_columns, fill_value=0)
    article = base.join(build_article_features(frame, primary, complementary))
    return base, article

## Обучение ранкеров на Train

In [7]:
PARAMS = {
    "n_estimators": 300,
    "learning_rate": 0.035,
    "num_leaves": 11,
    "min_child_samples": 80,
    "reg_lambda": 20,
    "lambdarank_truncation_level": 10,
    "random_state": 42,
    "n_jobs": 2,
    "verbosity": -1,
}


def fit_branch(frame, base_features, article_features):
    query_ids = frame.query_id.to_numpy()
    rows = np.argsort(query_ids, kind="stable")
    _, group_sizes = np.unique(query_ids[rows], return_counts=True)
    labels = frame.relevance.iloc[rows].to_numpy(dtype=np.int32)

    models = []
    for objective, features in (
        ("lambdarank", base_features),
        ("rank_xendcg", article_features),
    ):
        model = LGBMRanker(objective=objective, **PARAMS)
        with threadpool_limits(limits=2):
            model.fit(features.iloc[rows], labels, group=group_sizes.tolist())
        models.append(model)

    return tuple(models)


train_base, train_article = build_branch(train, train_forward, train_reverse)
train_mirror_base, train_mirror_article = build_branch(
    train, train_reverse, train_forward
)

lm, xe = fit_branch(train, train_base, train_article)
mirror_lm, mirror_xe = fit_branch(train, train_mirror_base, train_mirror_article)

## Ранжирование Test

In [8]:
def z_by_query(frame, scores):
    values = pd.Series(np.asarray(scores, dtype=float))
    group = values.groupby(frame.query_id.to_numpy(), sort=False)
    centered = values - group.transform("mean")
    scale = np.maximum(group.transform("std").fillna(0), 1e-6)
    return (centered / scale).to_numpy()



def direction_scores(frame, complementary_jina, lm, xe):
    jina = 0.25 * z_by_query(frame, complementary_jina)
    head = jina + 0.75 * z_by_query(frame, lm)
    article = jina + 0.75 * z_by_query(frame, xe)
    return head, (head + article) / 2


def make_ranks(frame, head, tail, fixed_rank=None):
    ids = frame.id.to_numpy(dtype=str)
    ranks = np.empty(len(frame), dtype=int)

    for rows in frame.groupby("query_id", sort=False).indices.values():
        if fixed_rank is None:
            leader = rows[np.lexsort((ids[rows], -head[rows]))][0]
        else:
            leader = rows[np.argmin(fixed_rank[rows])]

        ordered = rows[np.lexsort((ids[rows], -tail[rows]))]
        ranked = np.r_[leader, ordered[ordered != leader]]
        ranks[ranked] = np.arange(1, len(rows) + 1)

    return ranks



test_forward, test_reverse = jina_scores(test, jina, "Test")
del jina
gc.collect()
torch.cuda.empty_cache()

test_base, test_article = build_branch(test, test_forward, test_reverse, train_base.columns)
test_mirror_base, test_mirror_article = build_branch(
    test, test_reverse, test_forward, train_mirror_base.columns
)

orig_head, orig_tail = direction_scores(
    test, test_reverse, lm.predict(test_base), xe.predict(test_article)
)
mirror_head, mirror_tail = direction_scores(
    test, test_forward, mirror_lm.predict(test_mirror_base), mirror_xe.predict(test_mirror_article)
)

rank_e30 = make_ranks(test, orig_head, orig_tail)
mixed_head = (orig_head + mirror_head) / 2
mixed_tail = (orig_tail + mirror_tail) / 2
rank_full = make_ranks(test, mixed_head, mixed_tail)
rank_tail = make_ranks(test, orig_head, mixed_tail, fixed_rank=rank_e30)

sizes = test.groupby("query_id", sort=False).id.transform("size").to_numpy()
rank_gated = np.where(sizes == 10, rank_full, rank_tail)

Jina Test (прямой порядок):   0%|          | 0/1305 [00:00<?, ?запрос/s]

Jina Test (обратный порядок):   0%|          | 0/1305 [00:00<?, ?запрос/s]

## Сабмит

In [9]:
prediction = pd.Series(1.0 / rank_gated, index=test.id)
submission_ids = test[["id"]]
submission = submission_ids.assign(prediction=submission_ids.id.map(prediction))
if not np.isfinite(submission.prediction.to_numpy()).all():
    raise ValueError("Для части строк отсутствуют предсказания.")

output_path = OUTPUT_DIR / "submission.csv"
submission.to_csv(output_path, index=False)
print(f"Сохранён {output_path.name}: {len(submission):,} строк.")

Сохранён submission.csv: 13,507 строк.


In [10]:
files.download(str(output_path))

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>